# Analyze repository links from AI-extracted paper metadata (links-only)

This notebook focuses on **dataset/code repository discovery** from the API extraction output in `data/from_papers/madata_results_api_2026-06_flat.csv`.

Key choices:
- We **only scan link-relevant columns** (strict schema) to avoid boilerplate links (e.g. Creative Commons, publisher sites).
- We extract **URLs and DOIs**, normalize them, classify known repository hosts, and export (tagged `2026-06`):
  - `extracted_repositories_2026-06.csv`
  - `extracted_repositories_with_papers_2026-06.csv`
  - `unknown_domains_2026-06.csv`
  - `unknown_links_2026-06.csv`


In [1]:
from __future__ import annotations

import re
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import plotly.express as px

try:
    from IPython.display import display
except ImportError:
    display = print  # noqa: A001


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for base in (here, here.parent):
        if (base / "data" / "from_papers").is_dir():
            return base
    return here


REPO = find_repo_root()
DATA = REPO / "data" / "from_papers"

# Input extraction file and output suffix (keep in sync with extract_metadata_api --output-tag)
EXTRACTION_CSV = "madata_results_api_2026-06_flat.csv"
OUTPUT_TAG = "2026-06"

print("Repo root:", REPO)
print("Data dir:", DATA)
print("Input:", EXTRACTION_CSV)
print("Output tag:", OUTPUT_TAG)


Repo root: C:\Users\student\Desktop\Madabi\Madabi
Data dir: C:\Users\student\Desktop\Madabi\Madabi\data\from_papers
Input: madata_results_api_2026-06_flat.csv
Output tag: 2026-06


In [2]:
# Regex: http(s) URLs and common DOI forms
_URL_RE = re.compile(
    r"https?://[^\s\]\)\"'<>\{\},]+|(?:doi:)?\s*(10\.\d{4,9}/\S+)",
    re.IGNORECASE,
)
_OSF_DOI_RE = re.compile(r"10\.17605/OSF\.IO/([A-Z0-9]+)", re.IGNORECASE)
_ZW_CHARS_RE = re.compile(r"[\u200B-\u200F\u202A-\u202E\u2060\uFEFF]")
# PDF extraction often inserts spaces inside hostnames, e.g. "https://osf. io/abc/".
_SPACED_OSF_HOST_RE = re.compile(r"(?i)(https?://)osf\s*\.\s*io")
_SPACED_OSF_HOST_NO_DOT_RE = re.compile(r"(?i)(https?://)osf\s+io")


def normalize_link_text(s: str) -> str:
    s = _ZW_CHARS_RE.sub("", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def _fix_spaced_url_hosts(s: str) -> str:
    """Repair line-break / OCR spaces inside known repository hostnames."""
    s = _SPACED_OSF_HOST_RE.sub(r"\1osf.io", s)
    s = _SPACED_OSF_HOST_NO_DOT_RE.sub(r"\1osf.io", s)
    return s


def classify_repository(url: str) -> str:
    url = normalize_link_text(str(url))
    u = url.lower()
    if "osf.io" in u or "open science framework" in u:
        return "OSF"
    if "github.com" in u:
        return "GitHub"
    if "gitlab" in u:
        return "GitLab"
    if "dataverse" in u or "/dvn/" in u or "10.7910/dvn" in u or "doi.org/10.7910" in u:
        return "Dataverse"
    if "figshare" in u or "10.6084/m9.figshare" in u:
        return "Figshare"
    if "zenodo" in u or "10.5281/zenodo" in u:
        return "Zenodo"
    if "gesis.org" in u or "search.gesis" in u:
        return "GESIS"
    if "madata.bib.uni-mannheim" in u or ("madata" in u and "uni-mannheim" in u):
        return "MADATA"
    if "doi.org" in u:
        return "DOI (other)"
    return "Other / unknown"


_TRAILING_JUNK = ").,;]\\\"'<>}"


def extract_urls_from_cell(val) -> list[str]:
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return []

    text = _fix_spaced_url_hosts(normalize_link_text(str(val)))
    if text.strip() in {"", "—", "-", "nan", "None"}:
        return []

    found: list[str] = []
    for m in _URL_RE.finditer(text):
        raw = normalize_link_text(m.group(0)).strip()
        raw = raw.replace(" ", "")

        if raw.lower().startswith("doi:") or re.match(r"^10\.\d", raw):
            raw = raw.replace("doi:", "").strip()
            if raw.startswith("10."):
                doi = raw.split()[0].rstrip(_TRAILING_JUNK)
                found.append(f"https://doi.org/{doi}")
            continue

        if raw.startswith("http"):
            found.append(raw.rstrip(_TRAILING_JUNK))

    for m in _OSF_DOI_RE.finditer(text):
        id_ = m.group(1)
        found.append(f"https://osf.io/{id_.lower()}/")

    return list(dict.fromkeys(found))


def extract_domain(url: str) -> str | None:
    url = normalize_link_text(str(url))
    try:
        host = urlparse(url).netloc.lower().strip()
    except Exception:
        return None
    if not host:
        return None
    if host.startswith("www."):
        host = host[4:]
    return host


In [3]:
LINK_COLS = [
    "13 Data DOI",
    "15 Data availability statement",
    "17 Code citation",
]


def _assert_expected_columns(cols) -> None:
    expected = ["filename", *LINK_COLS]
    missing = [c for c in expected if c not in set(cols)]
    if missing:
        raise KeyError(
            f"Missing expected columns in {EXTRACTION_CSV}: "
            + ", ".join(missing)
            + ". Available columns: "
            + ", ".join(map(str, cols))
        )


def load_extraction_table() -> pd.DataFrame:
    """Load only link-relevant columns from the 2026-06 flat extraction CSV."""

    csv_path = DATA / EXTRACTION_CSV
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing {csv_path.name} in {DATA}")

    header = pd.read_csv(csv_path, nrows=0)
    _assert_expected_columns(header.columns)

    return pd.read_csv(csv_path, usecols=["filename", *LINK_COLS])


def columns_to_scan(df: pd.DataFrame) -> list[str]:
    _assert_expected_columns(df.columns)
    return LINK_COLS


def _is_present(v) -> bool:
    if v is None:
        return False
    if isinstance(v, float) and pd.isna(v):
        return False
    if isinstance(v, str) and v.strip() == "":
        return False
    return True


def explode_urls(df: pd.DataFrame, label: str) -> pd.DataFrame:
    rows: list[dict] = []
    cols = columns_to_scan(df)
    for _, row in df.iterrows():
        fname = row.get("filename", "")
        combined = " ".join(str(row[c]) for c in cols if _is_present(row[c]))
        for url in extract_urls_from_cell(combined):
            rows.append(
                {
                    "source_table": label,
                    "filename": fname,
                    "url": url,
                    "repository": classify_repository(url),
                }
            )
    return pd.DataFrame(rows)


extraction_df = load_extraction_table()
print(f"extraction table ({EXTRACTION_CSV}): {len(extraction_df)} papers × {len(extraction_df.columns)} cols")
urls_long = explode_urls(extraction_df, f"2026-06 ({EXTRACTION_CSV})")
urls_long = urls_long.drop_duplicates(subset=["filename", "url"]) if not urls_long.empty else urls_long

DATASET_REPOS = {"OSF", "GitHub", "GitLab", "Dataverse", "Zenodo", "Figshare", "GESIS", "MADATA", ""}
repos_long = (
    urls_long[urls_long["repository"].isin(DATASET_REPOS)].copy()
    if not urls_long.empty
    else pd.DataFrame(columns=["source_table", "filename", "url", "repository"])
)

print(f"All extracted links: {len(urls_long)} | Dataset/replication repos: {len(repos_long)}")
display(repos_long.head(20))


extraction table (madata_results_api_2026-06_flat.csv): 2063 papers × 4 cols
All extracted links: 1200 | Dataset/replication repos: 707


,source_table,filename,url,repository
1,2026-06 (madata_results_api_2026-06_flat.csv),00220027221104719.pdf,https://dataverse.harvard.edu/dataverse/sabine...,Dataverse
9,2026-06 (madata_results_api_2026-06_flat.csv),0038-6073-2025-1-2-105.pdf,https://osf.io/z84d5/,OSF
12,2026-06 (madata_results_api_2026-06_flat.csv),0049124119882480.pdf,https://dbk.gesis.org/dbksearch/GDesc2.asp?no¤...,GESIS
13,2026-06 (madata_results_api_2026-06_flat.csv),0093650220939778.pdf,https://osf.io/s3b5z/,OSF
14,2026-06 (madata_results_api_2026-06_flat.csv),0093650220944790.pdf,https://doi.org/10.17605/osf.io/uc359,OSF
15,2026-06 (madata_results_api_2026-06_flat.csv),0093650220944790.pdf,https://osf.io/uc359/,OSF
17,2026-06 (madata_results_api_2026-06_flat.csv),01461672211031080_.pdf,https://osf.io/mvpsw/?view_only=06aa074594334f...,OSF
18,2026-06 (madata_results_api_2026-06_flat.csv),01_CCR2021.1_CHAN.pdf,https://doi.org/10.17605/OSF.IO/UTXS5,OSF
19,2026-06 (madata_results_api_2026-06_flat.csv),01_CCR2021.1_CHAN.pdf,https://github.com/chainsawriot/ots/,GitHub
20,2026-06 (madata_results_api_2026-06_flat.csv),01_CCR2021.1_CHAN.pdf,https://osf.io/utxs5/,OSF


In [5]:
# Summary plot + exports
n = len(repos_long)
if n == 0:
    print("No dataset/replication repository links found.")
else:
    repo_counts = (
        repos_long["repository"]
        .value_counts()
        .rename_axis("repository")
        .reset_index(name="count")
    )
    display(repo_counts)

    fig = px.bar(
        repo_counts,
        x="repository",
        y="count",
        title=f"Dataset/replication repository links ({OUTPUT_TAG}, deduped filename+URL)",
        labels={"count": "Count", "repository": "Type"},
    )
    fig.update_xaxes(tickangle=-35)
    fig.show()

output_unique = DATA / f"extracted_repositories_{OUTPUT_TAG}.csv"
output_papers = DATA / f"extracted_repositories_with_papers_{OUTPUT_TAG}.csv"

if n > 0:
    unique_repos = (
        repos_long.groupby(["repository", "url"], as_index=False)
        .agg(distinct_papers=("filename", "nunique"), mention_rows=("filename", "size"))
        .sort_values(["repository", "distinct_papers", "mention_rows"], ascending=[True, False, False])
    )
    paper_links = (
        repos_long[["filename", "repository", "url"]]
        .drop_duplicates()
        .sort_values(["repository", "filename", "url"])
    )
    unique_repos.to_csv(output_unique, index=False)
    paper_links.to_csv(output_papers, index=False)
    print(f"Wrote {len(unique_repos)} unique repositories to: {output_unique}")
    print(f"Wrote {len(paper_links)} paper-level rows to: {output_papers}")


,repository,count
0,OSF,421
1,Dataverse,128
2,GitHub,100
3,Zenodo,27
4,MADATA,11
5,Figshare,10
6,GESIS,5
7,GitLab,5


Wrote 697 unique repositories to: C:\Users\student\Desktop\Madabi\Madabi\data\from_papers\extracted_repositories_2026-06.csv
Wrote 707 paper-level rows to: C:\Users\student\Desktop\Madabi\Madabi\data\from_papers\extracted_repositories_with_papers_2026-06.csv


In [6]:
# Unknown links: rank domains and write review CSVs
if urls_long.empty:
    print("No links extracted; skip unknown-domain export.")
else:
    u = urls_long[urls_long["repository"] == "Other / unknown"].copy()
    u["domain"] = u["url"].map(extract_domain)
    unknown = u[u["domain"].notna()].copy()
    print(f"unknown links: {len(unknown)} | distinct papers: {unknown['filename'].nunique()}")
    display(unknown[["filename", "url", "repository", "domain"]].head(10))

    by_domain = (
        unknown.groupby("domain", as_index=False)
        .agg(
            link_mentions=("url", "size"),
            unique_links=("url", "nunique"),
            distinct_papers=("filename", "nunique"),
        )
        .sort_values(["distinct_papers", "unique_links", "link_mentions"], ascending=[False, False, False])
    )
    display(by_domain.head(50))

    out_domains = DATA / f"unknown_domains_{OUTPUT_TAG}.csv"
    out_links = DATA / f"unknown_links_{OUTPUT_TAG}.csv"
    by_domain.to_csv(out_domains, index=False)
    unknown.sort_values(["domain", "filename"]).to_csv(out_links, index=False)
    print("Wrote:")
    print("-", out_domains)
    print("-", out_links)

    top_n = 25
    plot_df = by_domain.head(top_n).sort_values("distinct_papers", ascending=True)
    fig_domains = px.bar(
        plot_df,
        x="distinct_papers",
        y="domain",
        orientation="h",
        title=f"Top {top_n} unknown domains (by distinct papers mentioning them)",
        labels={"distinct_papers": "Distinct papers", "domain": ""},
        hover_data=["link_mentions", "unique_links"],
    )
    fig_domains.update_layout(yaxis_title="", margin=dict(l=10, r=20, t=50, b=40), height=max(420, top_n * 18))
    fig_domains.show()


unknown links: 172 | distinct papers: 141


,filename,url,repository,domain
0,00027642211021646.pdf,http://www.worldvaluesurvey,Other / unknown,worldvaluesurvey
5,0022343318800524.pdf,http://www.prio.org/jpr/datasets,Other / unknown,prio.org
6,0022343319897105.pdf,http://www.prio.org/jpr/datasets,Other / unknown,prio.org
25,0894439318816389.pdf,https://www.iab.de/en/daten.aspx,Other / unknown,iab.de
26,0894439320914261.pdf,https://ukdataservice.ac.uk/,Other / unknown,ukdataservice.ac.uk
28,0894439320944118.pdf,https://www.iab.de/en/daten.aspx,Other / unknown,iab.de
30,08944393211032950.pdf,https://www.iab.de/en/daten.aspx,Other / unknown,iab.de
33,0956797620951115.pdf,http://journals.sagepub.com/doi/suppl/10.1177/...,Other / unknown,journals.sagepub.com
37,1-s2.0-S0005796725000130-main.pdf,https://doi,Other / unknown,doi
38,1-s2.0-S0005796725002748-main.pdf,https://doi,Other / unknown,doi


,domain,link_mentions,unique_links,distinct_papers
89,osf,10,2,10
28,doi,9,1,9
27,diw.de,6,5,5
64,journals.sagepub.com,4,4,4
31,ec.europa.eu,3,3,3
111,uni-mannheim.de,3,3,3
115,worldvaluessurvey.org,3,3,3
91,pairfam.de,3,2,3
97,prio.org,3,2,3
103,share-project.org,3,2,3


Wrote:
- C:\Users\student\Desktop\Madabi\Madabi\data\from_papers\unknown_domains_2026-06.csv
- C:\Users\student\Desktop\Madabi\Madabi\data\from_papers\unknown_links_2026-06.csv


In [7]:
unknown[unknown["domain"] == "creativecommons.org"][["filename", "url"]].head(20)

,filename,url
